# FIT5230 Milestone 3: Full Native Unicode Attack Experiments

**Team:** Dark.M&M  
**Members:** Chaoyue Du; Weijun Meng  
**Individual contributions:** [Complete with the real division of work before submission]

This is our own M3 experiment notebook, not the peer-team case study.
It loads the original PIGuard model/checkpoint, executes attacks and controls, and generates results from new model predictions.
Read and run top to bottom. The real model and main comparisons run by default; extension experiments have explicit switches.
No historical success counts are printed as if freshly obtained. Disabled extensions remain visible but make no model calls or result claims.


## 1. Problem and development from M1/M2

PIGuard classifies text before a downstream assistant: `0 = benign`, `1 = injection`.
Our attack changes Unicode representation without retraining the guard. M3 extends the earlier Unicode work by comparing word selection, joint edits, character families, query costs, false blocks and another detector.

Research questions:
1. Does gradient-guided word ranking help under the same candidate budget?
2. Can joint edits find misses that the tested single-edit method misses?
3. Which Unicode family gives the best coverage/cost in the matched experiment?
4. Do changes also cause benign false blocks, and what do repair controls show?
5. Do source-successful strings transfer, or must each detector be searched separately?

These are separate protocols. Their success counts are not one continuous improvement curve.


### Full workflow and runtime

**Default main workflow:** environment → datasets → original model/checkpoint → real example predictions → combination controls → two-model Unicode family comparison → current-run conclusions.

**Optional extensions:** gradient ranking, paid-probe selection, full stability/repair, and frozen direct transfer. They are OFF by default; enable them in Section 4.1 when needed.

Use **T4 GPU** in Colab. Main experiments keep their full frozen cohorts, budgets and controls; no easier samples are substituted.
The optional full stability experiment includes 464 originals and 16,101 variants plus repair controls. It is NOT run by default.
Model downloads and the selected main comparisons still take time. Exact duration depends on Colab resources.
Source/data files are downloaded from our public GitHub repository at a fixed commit, then checked by SHA256. No local upload or encoded package is needed. Model weights are downloaded separately.


In [ ]:
# Download the fixed M3 source/data from our GitHub repository.
import hashlib
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path('/content/FIT5230_M3_Main_Optional')
SOURCE = ROOT / "src"
SOURCE_REVISION = '31ca5dc1ff1a27aeb05c6b2de663a31f7e616093'
REPO_URL = 'https://github.com/dcy1279-hash/PIGuard.git'
CHECKOUT = ROOT.parent / ("FIT5230_M3_Source_" + SOURCE_REVISION[:12])
if not CHECKOUT.exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
    subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--quiet", "--detach", SOURCE_REVISION], check=True)
head = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip()
assert head == SOURCE_REVISION, "Source checkout differs; preserve it and use a fresh CHECKOUT."

release = CHECKOUT / "m3"
inventory_bytes = (release / "inventory.json").read_bytes()
assert hashlib.sha256(inventory_bytes).hexdigest() == '97c33457c09df81e7da2758da060eec982b515c8ec406e3b2224fa98380d73d9', "Source inventory differs."
inventory = json.loads(inventory_bytes)
contents = {"inventory.json": inventory_bytes}
for name, expected in inventory["files_sha256"].items():
    assert not Path(name).is_absolute() and ".." not in Path(name).parts, "Unsafe source path."
    content = (release / name).read_bytes()
    assert hashlib.sha256(content).hexdigest() == expected, "Source/data differs: " + name
    contents[name] = content
for name, content in contents.items():
    destination = ROOT / name
    if destination.exists():
        assert destination.read_bytes() == content, "Existing source differs; preserve it and choose a fresh ROOT."
for name, content in contents.items():
    destination = ROOT / name
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)

sys.path.insert(0, str(SOURCE))
cached_driver = sys.modules.get("m3_full_runtime")
if cached_driver is not None and Path(cached_driver.__file__).resolve() != (SOURCE / "m3_full_runtime.py").resolve():
    del sys.modules["m3_full_runtime"]
from IPython.display import Code, Markdown, display
from IPython.core.magic import register_cell_magic
from m3_submission import show_table

@register_cell_magic
def verified_source(filename, body):
    # Check visible code without changing the author's runtime bytes.
    name = filename.strip()
    assert Path(name).name == name
    original = (SOURCE / name).read_text(encoding="utf-8")
    normalize = lambda text: "\n".join(line.rstrip() for line in text.splitlines()).strip()
    assert normalize(body) == normalize(original), "Visible code differs from the runtime source"
    print("Verified visible implementation:", SOURCE / name)

print("M3 source, data and frozen cohorts verified from GitHub:", SOURCE_REVISION)
print("Model weights are downloaded separately. Results require real model inference.")


## 2. Original PIGuard model

The class below is the unchanged author's implementation: DeBERTa encoder plus a two-class linear head.
The author checkpoint will be loaded strictly; we do not train a new guard or tune its threshold.
Its native `classify` path uses a 256-token window and argmax.
`verified_source` checks that the visible implementation matches the file actually used by the native runner.


In [ ]:
%%verified_source PIGuard.py
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, AutoConfig

class PIGuard(nn.Module):
    def __init__(self, model_name, num_labels, device):
        super(PIGuard, self).__init__()
        self.device = device
        self.config = AutoConfig.from_pretrained(model_name, output_attentions=True)
        self.deberta = AutoModel.from_pretrained(model_name, config=self.config).to(device)
        self.classifier = nn.Linear(self.deberta.config.hidden_size, num_labels).to(device)
        # self.lm_head = nn.Linear(self.deberta.config.hidden_size, self.deberta.config.vocab_size).to(device)
        self.loss_fct = nn.CrossEntropyLoss().to(device)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)


    def forward(self, input_ids, attention_mask, labels=None, mode="train"):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        last_hidden_state = outputs['last_hidden_state']
        # sequence_output = last_hidden_state
        pooled_output = last_hidden_state[:, 0, :]

        logits = self.classifier(pooled_output)

        loss = 0
        loss = self.loss_fct(logits.view(-1, self.classifier.out_features), labels.view(-1))

        return logits, loss


    def classify(self, input_text):
        tokenzied_text = self.tokenizer(input_text, return_tensors='pt', truncation=True, padding='max_length', max_length=256)
        input_ids = tokenzied_text['input_ids'].to(self.device)
        attention_mask = tokenzied_text['attention_mask'].to(self.device)
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)['last_hidden_state']
        sequence_output = outputs
        pooled_output = outputs[:, 0, :]

        classification_logits = self.classifier(pooled_output)
        return classification_logits


## 3. Our attack implementations

### 3.1 Label/score-visible combination engine

The complete implementation is visible below, not only a diagram or a result table.
The attack receives an oracle returning a label and injection score; it does not receive the model or tokenizer.
Original checks, failures and limited removal checks all count toward the fixed 30-query budget.


In [ ]:
%%verified_source unicode_combo_attack.py
"""Model-independent score-oracle Unicode search. No tensor/model/gradient access.

The only target information accepted by attack() is oracle(text)->(label,p1).
Single edits probe positions, then a width-two beam combines up to three edits.
Controls use the same anchors/stage caps but never rank nodes by target scores.
"""
from itertools import combinations, zip_longest
import hashlib
import math
import random

from eval_m3_unicode_stability import CYRILLIC, eligible_words

MAX_QUERIES = 30
MAX_EDITS = 3
ANCHORS = 8
STAGE_LIMITS = (8, 8, 10)
REFINEMENT_QUERIES = 3
BEAM_WIDTH = 2
SEARCH_QUERY_CAP = MAX_QUERIES - REFINEMENT_QUERIES  # includes original query
RANDOM_SEEDS = tuple(range(5230, 5240))
STRATEGIES = ("single_only", "sequential_combo", "feedback_combo") + tuple(f"random_combo_{s}" for s in RANDOM_SEEDS)
CONFIG = dict(max_total_queries=MAX_QUERIES, max_edited_codepoints=MAX_EDITS,
    original_query_counted=True, max_refinement_queries=REFINEMENT_QUERIES,
    search_cap_including_original=SEARCH_QUERY_CAP, anchors=ANCHORS,
    stage_candidate_caps=list(STAGE_LIMITS), beam_width=BEAM_WIDTH,
    position_order="round robin words left-to-right, mappable letters within word left-to-right",
    mutation="fixed Cyrillic confusables; distinct positions; reversible substitutions only",
    combo_support="same first up-to-eight round-robin positions for all combination methods",
    feedback="rank observed unsuccessful nodes by injection probability, lower first; ties positions",
    random_seeds=list(RANDOM_SEEDS), random_order="SHA256(seed:sample_id:depth) local RNG",
    single_control="up to 29 round-robin single substitutions, no combination; may explore outside anchors",
    refinement="at most three deletion-of-edit checks; no claim of global minimum",
    threat_model="simulated score-visible black box; label+probability only, no gradients/weights",
    independent_holdout=False)


def positions(text):
    """Breadth across words before revisiting a word; text-only static scope."""
    words = [[a+i for i,c in enumerate(token) if c in CYRILLIC]
             for a,b,token in eligible_words(text)]
    return [p for group in zip_longest(*words) for p in group if p is not None]


def apply_edits(text, edits):
    edits=tuple(sorted(edits))
    legal=set(positions(text))
    if len(set(edits))!=len(edits) or len(edits)>MAX_EDITS or not set(edits)<=legal:
        raise ValueError("Invalid/protected/duplicate Unicode edit")
    chars=list(text)
    for pos in edits: chars[pos]=CYRILLIC[text[pos]]
    changed="".join(chars)
    if sum(a!=b for a,b in zip(text,changed))!=len(edits) or len(changed)!=len(text):
        raise AssertionError("Unexpected text transformation")
    if changed.translate(str.maketrans({v:k for k,v in CYRILLIC.items()}))!=text:
        raise ValueError("Original must be ASCII for exact reversibility")
    return changed


def random_order(nodes, seed, sid, depth):
    nodes=list(nodes)
    local=int.from_bytes(hashlib.sha256(f"{seed}:{sid}:{depth}".encode()).digest()[:8],"big")
    random.Random(local).shuffle(nodes)
    return nodes


def children(parents, anchors):
    """Round robin two parents, deduplicated canonical edit sets."""
    lists=[[(tuple(sorted((*p,a))),p) for a in anchors if a not in p] for p in parents]
    seen=set();result=[]
    for group in zip_longest(*lists):
        for item in group:
            if item is None: continue
            node,parent=item
            if node not in seen:
                seen.add(node);result.append((node,parent))
    return result


def attack(text, oracle, strategy, sample_id="input"):
    if strategy not in STRATEGIES: raise ValueError("Unknown frozen strategy")
    if not text.isascii(): raise ValueError("Frozen experiment requires ASCII originals")
    support=positions(text);anchors=support[:ANCHORS]
    if len(support)<3: raise ValueError("Insufficient candidate positions")
    seed=int(strategy.rsplit("_",1)[1]) if strategy.startswith("random_combo_") else None
    trace=[];winner=None;first_success=0;best=None

    def query(edits,phase,parent=()):
        nonlocal best
        if len(trace)>=MAX_QUERIES: raise RuntimeError("Total query cap exceeded")
        current=apply_edits(text,edits)
        label,probability=oracle(current)
        if label not in (0,1) or not math.isfinite(probability) or not 0<=probability<=1:
            raise ValueError("Invalid score-visible oracle response")
        row=dict(query_index=len(trace)+1,phase=phase,depth=len(edits),
            positions=list(edits),parent_positions=list(parent),text=current,
            prediction=int(label),injection_probability=float(probability))
        trace.append(row)
        if best is None or (probability,len(edits),tuple(edits))<(best["injection_probability"],best["depth"],tuple(best["positions"])):
            best=row
        return row

    original=query((),"original")
    if original["prediction"]!=1:
        raise RuntimeError("Original no longer detected: stop, do not count as new success")
    if strategy=="single_only":
        for pos in support[:MAX_QUERIES-1]:
            row=query((pos,),"single")
            if row["prediction"]==0:
                winner=row;first_success=row["query_index"];break
    else:
        previous=[]
        for depth,cap in enumerate(STAGE_LIMITS,1):
            if depth==1:
                proposals=[((p,),()) for p in anchors]
            elif strategy=="feedback_combo":
                ranked=sorted(previous,key=lambda r:(r["injection_probability"],tuple(r["positions"])))[:BEAM_WIDTH]
                proposals=children([tuple(r["positions"]) for r in ranked],anchors)
            else:
                proposals=[(tuple(sorted(c)),()) for c in combinations(anchors,depth)]
            if seed is not None:
                proposals=random_order(proposals,seed,sample_id,depth)
            observed=[]
            for edits,parent in proposals[:cap]:
                if len(trace)>=SEARCH_QUERY_CAP: break
                row=query(edits,"single" if depth==1 else "combination",parent)
                observed.append(row)
                if row["prediction"]==0:
                    winner=row;first_success=row["query_index"];break
            if winner is not None: break
            previous=observed
            if not previous: break

    first_winner=winner
    refinement_calls=0
    if winner is not None and strategy!="single_only":
        # Removal order fixed once. A limited greedy reduction, not a proof of minimality.
        for pos in reversed(winner["positions"]):
            if len(winner["positions"])<=1 or refinement_calls>=REFINEMENT_QUERIES: break
            reduced=tuple(p for p in winner["positions"] if p!=pos)
            checked=query(reduced,"refinement",tuple(winner["positions"]))
            refinement_calls+=1
            if checked["prediction"]==0: winner=checked
    selected=winner if winner is not None else best
    result=dict(strategy=strategy,random_seed=seed,success=winner is not None,
        total_queries=len(trace),search_queries=len(trace)-refinement_calls,
        refinement_queries=refinement_calls,first_success_query=first_success,
        first_success_edits=first_winner["depth"] if first_winner else 0,
        first_success_positions=first_winner["positions"] if first_winner else [],
        first_success_text=first_winner["text"] if first_winner else "",
        final_edits=winner["depth"] if winner else 0,final_positions=winner["positions"] if winner else [],
        final_text=winner["text"] if winner else "",candidate_positions=len(support),
        anchor_positions=anchors,best_probability=best["injection_probability"],
        best_positions=best["positions"],best_text=best["text"],
        selected_probability=selected["injection_probability"],
        original_probability=original["injection_probability"],
        global_minimum_proven=False)
    return result,trace


### 3.2 Matched Unicode family engine

Same legal word-internal positions, at most three unit edits and 30 total queries.
One Cyrillic substitution and one U+200B insertion each cost one unit. Each candidate is rebuilt using original coordinates.
Protected code, quoted literals, URLs and email addresses are excluded. The unchanged engine includes feedback and sequential controls.


In [ ]:
%%verified_source unicode_mixed_attack.py
"""Matched-site Cyrillic / U+200B / mixed oracle-only Unicode search."""
import math
import unicode_revisit_attack as revisit

old=revisit.old
ZW='\u200b'
SEEDS=revisit.SEEDS
STRATEGIES=('single_reference',)+tuple(f'{family}_{policy}' for family in ('cyr','zw','mixed') for policy in ('feedback','sequential'))+tuple(f'mixed_random_{s}' for s in SEEDS)
CONFIG=dict(max_total_queries=30,search_cap_including_original=27,max_refinement_queries=3,
    max_unit_edits_per_input=3,substitution_cost=1,insertion_cost=1,
    max_one_action_per_original_site=True,zero_width_character='U+200B',
    shared_sites='original round-robin Cyrillic-mappable letters strictly inside eligible words (not final letter)',
    search='section14 revisitable skeleton, same restart/add/swap/drop cycle and parent expansion tiers',
    initial_probes='up to8 identical sites; pure uses own operator, mixed alternates cyr/zw by site index',
    mixed_restart_order='round-robin sites with cyr then zw at each site; skip observed states',
    controls='cyr/zw/mixed each feedback and sequential; mixed random5230-5239; unchanged old single reference has its original broader site pool',
    random_seeds=list(SEEDS),budgets=[5,10,20,30],independent_holdout=False,
    exploratory_after_prior_results=True,threat_model='label+score oracle only, no tokenizer/model/gradient')


def sites(text):
    words=old.eligible_words(text)
    return [p for p in old.positions(text) if any(a<=p and p+1<b for a,b,w in words)]


def canonical(actions):return tuple(sorted(tuple(a) for a in actions))


def render(text,actions,reference=False):
    actions=canonical(actions);legal=set(old.positions(text) if reference else sites(text))
    if not text.isascii() or len(actions)>3 or len({p for kind,p in actions})!=len(actions):raise ValueError('Invalid unit-edit state')
    if any(kind not in ('c','z') or p not in legal or (reference and kind!='c') for kind,p in actions):raise ValueError('Illegal/protected action')
    by={p:kind for kind,p in actions};pieces=[];meta=[];index=0
    for p,c in enumerate(text):
        kind=by.get(p)
        if kind=='c':
            pieces.append(old.CYRILLIC[c]);meta.append(dict(kind='c',original_position=p,start=index,end=index+1,required_spans=[[index,index+1]]));index+=1
        else:
            pieces.append(c);index+=1
            if kind=='z':
                pieces.append(ZW);meta.append(dict(kind='z',original_position=p,start=index,end=index+1,required_spans=[[index-1,index],[index+1,index+2]]));index+=1
    changed=''.join(pieces)
    if changed.replace(ZW,'').translate(str.maketrans({v:k for k,v in old.CYRILLIC.items()}))!=text:raise AssertionError('Not exactly reversible')
    assert len(changed)==len(text)+sum(k=='z' for k,p in actions)
    return changed,meta


def neighbors(parent,pool,operation,used):
    present=set(parent);occupied={p for kind,p in parent}
    additions=[a for a in pool if a not in used]+[a for a in pool if a in used]
    if operation=='add' and len(parent)<3:
        for a in additions:
            if a[1] not in occupied:yield canonical((*parent,a))
    elif operation=='swap' and len(parent)>1:
        for a in additions:
            if a in present:continue
            for removed in reversed(parent):
                reduced=present-{removed}
                if a[1] not in {p for k,p in reduced}:yield canonical((*reduced,a))
    elif operation=='drop' and len(parent)>1:
        for removed in reversed(parent):yield canonical(present-{removed})


def annotate(result,trace,text,family,policy):
    for row in trace:
        reference=family=='reference_cyr'
        actions=canonical(row.get('actions',[('c',p) for p in row['positions']]))
        rebuilt,meta=render(text,actions,reference)
        assert rebuilt==row['text']
        row.update(actions=[list(a) for a in actions],edit_metadata=meta,
            escaped_text=row['text'].encode('unicode_escape').decode('ascii'),
            cyr_edits=sum(k=='c' for k,p in actions),zw_edits=sum(k=='z' for k,p in actions))
    first=trace[result['first_success_query']-1] if result['success'] else None
    result.update(family=family,policy=policy,shared_sites=sites(text),
        allowed_sites=old.positions(text) if family=='reference_cyr' else sites(text),
        first_success_operation=first['phase'] if first else '',
        first_success_actions=first['actions'] if first else [],
        first_success_is_mixed=bool(first and first['cyr_edits'] and first['zw_edits']))
    return result,trace


def attack(text,oracle,strategy,sample_id='input'):
    if strategy not in STRATEGIES:raise ValueError('Unknown mixed strategy')
    if strategy=='single_reference':
        r,t=old.attack(text,oracle,'single_only',sample_id);r['strategy']=strategy
        r['final_actions']=[['c',p] for p in r['final_positions']]
        return annotate(r,t,text,'reference_cyr','reference')
    family,policy=strategy.split('_',1);seed=int(policy.rsplit('_',1)[1]) if policy.startswith('random_') else None
    support=sites(text)
    if not support:raise ValueError('No common legal site; do not silently resample')
    kinds=('c','z') if family=='mixed' else ('c',) if family=='cyr' else ('z',)
    pool=[(k,p) for p in support for k in kinds]
    initial=[(kinds[i%len(kinds)],p) for i,p in enumerate(support[:8])]
    trace=[];observed={};expansions={};best=None;winner=None;first_query=0;refinement=0
    def query(actions,phase,parent=(),requested='initial'):
        nonlocal best
        if len(trace)>=30:raise RuntimeError('Query budget exceeded')
        actions=canonical(actions);parent=canonical(parent);changed,meta=render(text,actions)
        label,prob=oracle(changed)
        if label not in (0,1) or not math.isfinite(prob) or not 0<=prob<=1:raise ValueError('Invalid oracle')
        prior=observed.get(parent)
        row=dict(query_index=len(trace)+1,phase=phase,requested_operation=requested,depth=len(actions),
            actions=[list(a) for a in actions],positions=sorted(p for k,p in actions),
            parent_actions=[list(a) for a in parent],parent_positions=sorted(p for k,p in parent),
            removed_actions=[list(a) for a in sorted(set(parent)-set(actions))],
            added_actions=[list(a) for a in sorted(set(actions)-set(parent))],
            parent_query_index=prior['query_index'] if prior else 0,parent_probability=prior['injection_probability'] if prior else None,
            text=changed,prediction=int(label),injection_probability=float(prob))
        trace.append(row)
        if best is None or (prob,len(actions),actions)<(best['injection_probability'],best['depth'],canonical(best['actions'])):best=row
        if phase!='refinement':observed[actions]=row
        return row
    original=query((),'original')
    if original['prediction']!=1:raise RuntimeError('Original label changed; stop, no resampling')
    for action in initial:
        row=query((action,),'single')
        if row['prediction']==0:winner=row;first_query=row['query_index'];break
    step=0;cycle=('restart','add','swap','drop')
    while winner is None and len(trace)<27:
        used={tuple(a) for row in trace for a in row['actions']};proposal=None
        requested=cycle[step%4]
        for operation in cycle[step%4:]+cycle[:step%4]:
            if operation=='restart':
                options=[(a,) for a in pool if (a,) not in observed]
                if options:proposal=(options[0],(),operation);break
            else:
                candidates=[]
                for parent,row in observed.items():
                    if not parent:continue
                    child=next((c for c in neighbors(parent,pool,operation,used) if c not in observed),None)
                    if child is not None:candidates.append((parent,row,child))
                if not candidates:continue
                tier=min(expansions.get((p,operation),0)//2 for p,r,c in candidates)
                candidates=[x for x in candidates if expansions.get((x[0],operation),0)//2==tier]
                if policy=='feedback':candidates.sort(key=lambda x:(x[1]['injection_probability'],len(x[0]),x[0]))
                elif policy=='sequential':candidates.sort(key=lambda x:x[1]['query_index'])
                else:candidates=old.random_order(candidates,seed,sample_id,100+step)
                parent,row,child=candidates[0];expansions[parent,operation]=expansions.get((parent,operation),0)+1
                proposal=(child,parent,operation);break
        if proposal is None:break
        actions,parent,operation=proposal;row=query(actions,operation,parent,requested);step+=1
        if row['prediction']==0:winner=row;first_query=row['query_index']
    first=winner
    if winner:
        for action in reversed(canonical(winner['actions'])):
            if winner['depth']<=1 or refinement>=3:break
            row=query(tuple(a for a in canonical(winner['actions']) if a!=action),'refinement',canonical(winner['actions']),'refinement');refinement+=1
            if row['prediction']==0:winner=row
    selected=winner if winner else best
    result=dict(strategy=strategy,random_seed=seed,success=winner is not None,total_queries=len(trace),
        search_queries=len(trace)-refinement,refinement_queries=refinement,first_success_query=first_query,
        first_success_edits=first['depth'] if first else 0,first_success_positions=first['positions'] if first else [],
        first_success_text=first['text'] if first else '',final_edits=winner['depth'] if winner else 0,
        final_positions=winner['positions'] if winner else [],final_actions=winner['actions'] if winner else [],
        final_text=winner['text'] if winner else '',final_is_mixed=bool(winner and {k for k,p in winner['actions']}=={'c','z'}),
        candidate_positions=len(support),best_probability=best['injection_probability'],best_positions=best['positions'],
        best_text=best['text'],selected_probability=selected['injection_probability'],original_probability=original['injection_probability'],
        global_minimum_proven=False)
    return annotate(result,trace,text,family,policy)


### 3.3 Paid-probe family selector

Try zero-width and Cyrillic at one common site; if both fail, continue the lower-scoring family.
The probes are paid, not free. Compare against fixed zero-width and a same-probe/always-zero control.
This is a hypothesis to test, not an assumed improvement.


In [ ]:
%%verified_source unicode_selector_attack.py
"""Two-probe family selection with ONE shared30-query budget.

No training, model ID, historical outcome, tokenization, or gradient input.
The chosen pure-family state machine resumes the already queried probe.
"""
import math
import unicode_mixed_attack as fixed

STRATEGIES=('zero_reference','cyr_reference','score_selector','paired_zero_control')
CONFIG=dict(max_total_queries=30,search_cap_including_original=27,max_refinement_queries=3,
    max_unit_edits_per_input=3,initial_probe_pairs=1,probe_order=['z','c'],
    probe_site='first static legal shared word-internal site; same site for both families',
    early_stop='any native label0 stops immediately, including a selection probe',
    selection='after two unsuccessful probes, lower observed injection probability; ties choose zero',
    continuation='resume selected immutable pure-family search order; saved own probe not sent again',
    control='same paired probes/early stopping, but force zero continuation if both probes unsuccessful',
    no_free_queries=True,no_model_id_or_historical_results_input=True,
    no_mixed_family_edit_sets=True,methods=list(STRATEGIES),budgets=[5,10,20,30],
    independent_holdout=False,exploratory_after_step1=True)


class FamilySearch:
    """Proposal-only equivalent of fixed pure feedback search, resumable."""
    def __init__(self,support,kind):
        self.kind=kind;self.pool=[(kind,p) for p in support];self.initial=self.pool[:8]
        self.observed={};self.expansions={};self.warm_index=0;self.step=0

    def observe(self,row):
        actions=fixed.canonical(row['actions'])
        if actions and all(k==self.kind for k,p in actions):self.observed[actions]=row

    def next_proposal(self):
        while self.warm_index<len(self.initial):
            action=self.initial[self.warm_index];self.warm_index+=1
            if (action,) not in self.observed:return (action,),(),'single','initial'
        cycle=('restart','add','swap','drop');requested=cycle[self.step%4]
        used={a for state in self.observed for a in state}
        for operation in cycle[self.step%4:]+cycle[:self.step%4]:
            if operation=='restart':
                option=next(((a,) for a in self.pool if (a,) not in self.observed),None)
                if option is not None:
                    self.step+=1;return option,(),operation,requested
            else:
                candidates=[]
                for parent,row in self.observed.items():
                    child=next((c for c in fixed.neighbors(parent,self.pool,operation,used) if c not in self.observed),None)
                    if child is not None:candidates.append((parent,row,child))
                if not candidates:continue
                tier=min(self.expansions.get((p,operation),0)//2 for p,r,c in candidates)
                candidates=[x for x in candidates if self.expansions.get((x[0],operation),0)//2==tier]
                parent,row,child=min(candidates,key=lambda x:(x[1]['injection_probability'],len(x[0]),x[0]))
                self.expansions[parent,operation]=self.expansions.get((parent,operation),0)+1
                self.step+=1;return child,parent,operation,requested
        return None


def attack(text,oracle,strategy,sample_id='input'):
    if strategy not in STRATEGIES:raise ValueError('Unknown selector strategy')
    if strategy in ('zero_reference','cyr_reference'):
        family='zw' if strategy=='zero_reference' else 'cyr'
        result,trace=fixed.attack(text,oracle,f'{family}_feedback',sample_id)
        result.update(strategy=strategy,chosen_family=family,choice_reason='fixed_reference',choice_after_query=0,
            selector_probe_queries=0,choice_probabilities={},own_probe_reused=False)
        return result,trace
    if not text.isascii():raise ValueError('ASCII originals required')
    support=fixed.sites(text)
    if not support:raise ValueError('No shared legal site; stop, no resampling')
    searches={k:FamilySearch(support,k) for k in ('z','c')}
    trace=[];observed={};best=None;winner=None;first_query=0;refinement=0
    chosen=None;reason='';choice_query=0;scores={}
    def query(actions,phase,parent=(),requested='initial',role='continuation'):
        nonlocal best
        if len(trace)>=30:raise RuntimeError('One global total-query cap exceeded')
        actions=fixed.canonical(actions);parent=fixed.canonical(parent)
        changed,meta=fixed.render(text,actions);label,prob=oracle(changed)
        if label not in (0,1) or not math.isfinite(prob) or not 0<=prob<=1:raise ValueError('Invalid label+probability')
        previous=observed.get(parent)
        row=dict(query_index=len(trace)+1,phase=phase,requested_operation=requested,role=role,
            depth=len(actions),actions=[list(a) for a in actions],positions=sorted(p for k,p in actions),
            parent_actions=[list(a) for a in parent],parent_positions=sorted(p for k,p in parent),
            removed_actions=[list(a) for a in sorted(set(parent)-set(actions))],
            added_actions=[list(a) for a in sorted(set(actions)-set(parent))],
            parent_query_index=previous['query_index'] if previous else 0,
            parent_probability=previous['injection_probability'] if previous else None,
            text=changed,prediction=int(label),injection_probability=float(prob))
        trace.append(row)
        if best is None or (prob,len(actions),actions)<(best['injection_probability'],best['depth'],fixed.canonical(best['actions'])):best=row
        if phase!='refinement':observed[actions]=row
        return row
    original=query((),'original',role='original')
    if original['prediction']!=1:raise RuntimeError('Original no longer detected; stop, no resampling')
    for kind in ('z','c'):
        row=query(((kind,support[0]),),'single',role='selection_probe')
        searches[kind].observe(row);scores[kind]=row['injection_probability']
        if row['prediction']==0:
            winner=row;first_query=row['query_index'];chosen=kind;reason='confirmed_probe_success';choice_query=len(trace);break
    if winner is None:
        chosen='z' if strategy=='paired_zero_control' or scores['z']<=scores['c'] else 'c'
        reason='forced_zero_after_same_probes' if strategy=='paired_zero_control' else 'lower_probe_probability_zero_on_tie'
        choice_query=len(trace)
        while len(trace)<27:
            proposal=searches[chosen].next_proposal()
            if proposal is None:break
            actions,parent,phase,requested=proposal
            row=query(actions,phase,parent,requested);searches[chosen].observe(row)
            if row['prediction']==0:winner=row;first_query=row['query_index'];break
    first=winner
    if winner:
        for action in reversed(fixed.canonical(winner['actions'])):
            if winner['depth']<=1 or refinement>=3:break
            row=query(tuple(a for a in fixed.canonical(winner['actions']) if a!=action),'refinement',fixed.canonical(winner['actions']),'refinement','refinement')
            refinement+=1
            if row['prediction']==0:winner=row
    selected=winner if winner else best
    family='zw' if chosen=='z' else 'cyr'
    result=dict(strategy=strategy,random_seed=None,success=winner is not None,total_queries=len(trace),
        search_queries=len(trace)-refinement,refinement_queries=refinement,first_success_query=first_query,
        first_success_edits=first['depth'] if first else 0,first_success_positions=first['positions'] if first else [],
        first_success_text=first['text'] if first else '',final_edits=winner['depth'] if winner else 0,
        final_positions=winner['positions'] if winner else [],final_actions=winner['actions'] if winner else [],
        final_text=winner['text'] if winner else '',final_is_mixed=False,candidate_positions=len(support),
        best_probability=best['injection_probability'],best_positions=best['positions'],best_text=best['text'],
        selected_probability=selected['injection_probability'],original_probability=original['injection_probability'],
        global_minimum_proven=False,chosen_family=family,choice_reason=reason,choice_after_query=choice_query,
        selector_probe_queries=sum(r.get('role')=='selection_probe' for r in trace),choice_probabilities=scores,
        own_probe_reused=reason!='confirmed_probe_success')
    return fixed.annotate(result,trace,text,family,'adaptive' if strategy=='score_selector' else 'paired_control')


### 3.4 White-box gradient word ranking

Differentiate the native class margin at input embeddings; average token-gradient norms within each fully visible word.
Rank words, then use the same fixed within-word character candidates as the sequential/random controls.
The gradient is a sensitivity heuristic, not an exact optimal Unicode edit. Parameters are not updated.


In [ ]:
import inspect
import eval_m3_gradient_pilot as gradient_core
display(Code(inspect.getsource(gradient_core.gradient_probe), language="python"))
display(Code(inspect.getsource(gradient_core.word_scores), language="python"))
display(Code(inspect.getsource(gradient_core.single_char_candidates), language="python"))


### 3.5 Native execution and result handling

The driver below launches the unchanged full protocols in a pinned environment.
The family stage calls the unchanged six-method engine on BOTH models with fresh predictions; it does not reuse old target outputs.
Completed stages can be reused within this session with an explicit message. Incomplete stages never fall back to old results.


In [ ]:
%%verified_source m3_full_runtime.py
"""Fresh-model execution and English result readers for the full M3 workbook.

Historical cohort definitions may be reused; historical predictions never act
as the oracle. Existing protocol sources, weights and budgets are unchanged.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys
import time
import traceback


TASKS = ("baseline", "stability", "gradient", "combination", "families", "selector", "transfer")
CORE_TASKS = ("baseline", "combination", "families")
OPTIONAL_TASKS = ("gradient", "selector", "stability", "transfer")


def selected_tasks(options=None):
    """Keep whole fixed protocols; switches select experiments, not easier samples."""
    options = {} if options is None else options
    if set(options) - set(OPTIONAL_TASKS) or any(type(value) is not bool for value in options.values()):
        raise ValueError("Optional experiment switches must be known Boolean values.")
    if options.get("transfer") and not options.get("stability"):
        raise ValueError("Direct transfer requires the current-run stability experiment. Enable RUN_STABILITY too.")
    return tuple(task for task in TASKS if task in CORE_TASKS or options.get(task, False))


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def csv_rows(path):
    with Path(path).open(newline="", encoding="utf-8-sig") as stream:
        return list(csv.DictReader(stream))


def verify_bundle(root):
    root = Path(root)
    inventory = read_json(root / "inventory.json")
    for name, expected in inventory["files_sha256"].items():
        if sha(root / name) != expected:
            raise RuntimeError(f"Source/data/plan differs from the fixed workbook: {name}")
    return inventory


def stream(command, cwd, logfile):
    """Stream real progress; keep complete legacy reports in the log, not the English display."""
    environment = {**os.environ, "MPLBACKEND": "Agg", "PYTHONUNBUFFERED": "1"}
    with Path(logfile).open("a", encoding="utf-8") as log:
        process = subprocess.Popen(list(map(str, command)), cwd=cwd, env=environment,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                # Existing benchmark text and source reports stay unchanged on disk.
                if not any("\u4e00" <= char <= "\u9fff" for char in line):
                    print(line, end="", flush=True)
            status = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if status:
        raise subprocess.CalledProcessError(status, command)


def result_files(directory):
    directory = Path(directory)
    return {str(path.relative_to(directory)): sha(path)
            for path in sorted(directory.rglob("*")) if path.is_file() and "__pycache__" not in path.parts}


def completed_result(session, task):
    session = Path(session)
    entry = read_json(session / "runs.json").get(task)
    if not entry or entry["status"] != "complete":
        raise RuntimeError(f"Run the fresh {task} stage first; old evidence is not a fallback.")
    path = Path(entry["result_dir"])
    if result_files(path) != entry["result_files_sha256"]:
        raise RuntimeError(f"Saved {task} result files changed or are missing.")
    return path


def run_stage(root, session, python, checkpoint, task, retry_incomplete=False):
    root, session, checkpoint = Path(root), Path(session), Path(checkpoint)
    if task not in TASKS:
        raise ValueError(task)
    verify_bundle(root)
    from eval_m3_unicode_stability import CHECKPOINT_SHA256
    if sha(checkpoint) != CHECKPOINT_SHA256:
        raise RuntimeError("Wrong author checkpoint.")
    session.mkdir(parents=True, exist_ok=True)
    identity = {"inventory_sha256": sha(root / "inventory.json"), "checkpoint_sha256": sha(checkpoint)}
    identity_file = session / "identity.json"
    if identity_file.exists() and read_json(identity_file) != identity:
        raise RuntimeError("Session belongs to different code, plans or weights. Use a new session.")
    write_json(identity_file, identity)
    index_path = session / "runs.json"
    index = read_json(index_path) if index_path.exists() else {}
    if task in index and index[task]["status"] == "complete":
        result = completed_result(session, task)
        print(f"{task}: reusing this session's completed native run; no repeated inference: {result}", flush=True)
        return result
    if task in index and not retry_incomplete:
        raise RuntimeError("An earlier attempt was incomplete. Its partial files are preserved. Set RETRY_INCOMPLETE=True only if you accept restarting this stage.")
    output = session / task / time.strftime("attempt_%Y%m%dT%H%M%S", time.gmtime())
    output.mkdir(parents=True, exist_ok=False)
    source, plans = root / "src", root / "plans"
    index[task] = {"status": "running", "attempt_dir": str(output), "fresh_model_inference": True}
    write_json(index_path, index)
    result = None
    try:
        if task in ("baseline", "families"):
            command = [python, "-u", source / "m3_full_runtime.py", "--worker", task,
                       "--root", root, "--checkpoint", checkpoint, "--output", output]
            stream(command, source, output / "run.log")
            result = output / "result"
        elif task in ("gradient", "combination", "selector"):
            script, member, prefix = {
                "gradient": ("eval_m3_gradient_text_full.py", "text_plan.json", "gradient_text_"),
                "combination": ("eval_m3_blackbox_combo.py", "combo_plan.json", "blackbox_combo_"),
                "selector": ("eval_m3_selector.py", "selector_plan.json", "selector_")}[task]
            raw = (plans / member).read_bytes()
            plan = output / member
            plan.write_bytes(raw)
            write_json(output / "preparation.json", {"plan_sha256": sha(plan)})
            stream([python, "-u", source / script, "--plan", plan, "--repo", source,
                    "--checkpoint", checkpoint, "--output-dir", output], source, output / "run.log")
            matches = list(output.glob(prefix + "*"))
            result = next((p for p in matches if (p / "manifest.json").is_file()
                           and read_json(p / "manifest.json").get("complete") is True), None)
        elif task == "stability":
            stream([python, "-u", source / "eval_m3_unicode_stability.py", "--dataset-root", root / "datasets",
                    "--resume", checkpoint, "--output-dir", output], source, output / "run.log")
            result = next((p for p in output.glob("stability_*") if (p / "manifest.json").is_file()
                           and read_json(p / "manifest.json").get("complete") is True), None)
        elif task == "transfer":
            stability = completed_result(session, "stability")
            stream([python, "-u", source / "eval_m3_transfer.py", "--source", stability,
                    "--output-dir", output], source, output / "run.log")
            frozen = next(output.glob("frozen_*/frozen_plan.json"))
            stream([python, "-u", source / "eval_m3_transfer.py", "--plan", frozen,
                    "--batch-size", "4", "--output-dir", output], source, output / "run.log")
            result = next((p for p in output.glob("transfer_*") if (p / "manifest.json").is_file()
                           and read_json(p / "manifest.json").get("complete") is True), None)
        if result is None or read_json(result / "manifest.json").get("complete") is not True:
            raise RuntimeError("No complete new result was generated. Do not display the historical result instead.")
        verify_bundle(root)
        index[task].update(status="complete", result_dir=str(result), result_files_sha256=result_files(result))
        write_json(index_path, index)
        print(f"FRESH {task} complete: {result}", flush=True)
        return result
    except BaseException:
        index[task].update(status="incomplete", error=traceback.format_exc())
        write_json(index_path, index)
        raise


def baseline_worker(root, checkpoint, output):
    import torch
    import eval_m3_gradient_pilot as pilot
    import eval_m3_unicode_stability as rules
    model, device = pilot.load_piguard(root / "src", checkpoint)
    print("PIGuard architecture and checkpoint loaded; actual device:", device, flush=True)
    records = rules.load_records(root / "datasets")
    original = read_json(root / "plans/text_plan.json")["selected"][0]["original"]
    candidates = [{"condition": "original", "text": original}]
    for family in rules.FAMILIES:
        candidate = next(iter(rules.candidates(original, family)))
        candidates.append({"condition": family, "text": candidate["text"]})
    rows = []
    versions = [(p, p._version) for p in model.parameters()]
    for spec in candidates:
        with torch.inference_mode():
            logits = model.classify(spec["text"])
            row = {**spec, "prediction": int(logits.argmax(-1).item()),
                   "injection_probability": float(logits.softmax(-1)[0, 1].cpu()),
                   "visible_text": spec["text"].replace("\u200b", "[U+200B]"),
                   "escaped_text": spec["text"].encode("unicode_escape").decode()}
        rows.append(row)
        print(row["condition"], "label", row["prediction"], "score", row["injection_probability"], flush=True)
    if any(p._version != version for p, version in versions):
        raise RuntimeError("Baseline model parameter changed.")
    result = output / "result"
    result.mkdir()
    write_json(result / "predictions.json", rows)
    write_json(result / "manifest.json", {"complete": True, "fresh_model_inference": True,
               "model": "microsoft/deberta-v3-base + original PIGuard classifier checkpoint",
               "checkpoint_sha256": sha(checkpoint), "model_source_sha256": sha(root / "src/PIGuard.py"),
               "device": str(device), "parameters": sum(p.numel() for p in model.parameters()),
               "native_window": 256, "dataset_records": len(records), "native_predictions": len(rows),
               "parameter_updates": 0, "scope": "Illustrative native predictions, not an attack-success benchmark."})


def family_worker(root, checkpoint, output):
    import gc
    import torch
    import eval_m3_complementarity as family
    import eval_m3_gradient_pilot as pilot
    plan = read_json(root / "plans/family_plan.json")
    result = output / "result"
    result.mkdir()
    all_outcomes, stats = [], {}
    for name, selected_key, baseline_key in (("piguard", "source_selected", "source_baseline"),
                                               ("protectai", "target_selected", "target_baseline")):
        directory = result / name
        directory.mkdir()
        if name == "piguard":
            model, device = pilot.load_piguard(root / "src", checkpoint)
            tokenizer, limit = model.tokenizer, 256
        else:
            tokenizer, model, device = family.reader.load_target()
            limit = 512
        print(f"Fresh matched families: {name}, device={device}, window={limit}", flush=True)
        versions = [(p, p._version) for p in model.parameters()]
        cache, baseline, outcomes, traces = {}, {}, [], []
        def predict(text):
            if text not in cache:
                with torch.inference_mode():
                    if name == "piguard":
                        logits = model.classify(text)
                    else:
                        encoded = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=limit)
                        logits = model(**{k: v.to(device) for k, v in encoded.items()}).logits
                    cache[text] = (int(logits.argmax(-1).item()), float(logits.softmax(-1)[0, 1].cpu()))
            return cache[text]
        for i, record in enumerate(plan["originals"], 1):
            label, probability = predict(record["original"])
            baseline[record["sample_id"]] = {"prediction": label, "probability": probability}
            if label != plan[baseline_key][record["sample_id"]]["prediction"]:
                raise RuntimeError(f"{name} original changed: {record['sample_id']}; stop without changing the denominator.")
            if i % 15 == 0:
                print(f"{name}: original checks {i}/75", flush=True)
        write_json(directory / "baseline_verification.json", baseline)
        for i, record in enumerate(plan[selected_key], 1):
            calls = []
            def oracle(text):
                fresh = text not in cache
                value = predict(text)
                calls.append(fresh)
                return value
            rows, log, count = family.evaluate_record(record, oracle,
                lambda text, metadata: family.audit_window(tokenizer, text, metadata, record["original"], limit))
            it = iter(calls)
            for row in log:
                local = row["physical_cache_hit"]
                native = False if local else next(it)
                row.update(local_cache_hit=local, physical_cache_hit=not native, native_inference_performed=native)
            if count != len(calls) or any(p._version != v for p, v in versions):
                raise RuntimeError("Native call ledger or parameter-integrity check failed.")
            outcomes.extend(dict(r, model=name) for r in rows)
            traces.extend(log)
            write_json(directory / "outcomes.json", outcomes)
            write_json(directory / "query_trace.json", traces)
            print(f"{name}: matched families {i}/{len(plan[selected_key])}", flush=True)
        stats[name] = {"device": str(device), "physical_model_inputs": len(cache),
                       "logical_attack_queries": len(traces), "parameter_updates": 0,
                       "new_model_inference": True, "records": len(plan[selected_key])}
        all_outcomes.extend(outcomes)
        del model, tokenizer, versions, cache, predict, oracle
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    budgets, complementary, costs, cross = family.analyse(plan, all_outcomes)
    for name, rows in (("all_outcomes.json", all_outcomes), ("budget_summary.json", budgets),
                       ("complementarity.json", complementary), ("paired_costs.json", costs), ("cross_model.json", cross)):
        write_json(result / name, rows)
    write_json(result / "manifest.json", {"complete": True, "fresh_model_inference": True,
               "protocol": "unchanged six matched-family strategies, fresh inference on BOTH detectors",
               "plan_sha256": sha(root / "plans/family_plan.json"), "stats": stats,
               "independent_holdout": False, "manual_semantic_review": "pending"})


def gradient_table(directory):
    rows, random_rows = csv_rows(Path(directory) / "budget_summary.csv"), csv_rows(Path(directory) / "random_seed_summary.csv")
    lookup = {(r["strategy"], int(r["budget"])): r for r in rows}
    return [{"Budget": int(r["budget"]), "Sequential": int(lookup["left_to_right", int(r["budget"])]["new_error_samples"]),
             "Gradient": int(lookup["gradient", int(r["budget"])]["new_error_samples"]),
             "Random_mean": float(r["mean_new_error_samples"]), "Denominator": int(r["denominator_per_seed"])} for r in random_rows]


def combination_table(directory):
    rows, random_rows = csv_rows(Path(directory) / "budget_summary.csv"), csv_rows(Path(directory) / "random_seed_summary.csv")
    lookup = {(r["strategy"], int(r["total_query_budget"])): r for r in rows}
    return [{"Budget": int(r["total_query_budget"]), "Single": int(lookup["single_only", int(r["total_query_budget"])]["audited_successes"]),
             "Feedback": int(lookup["feedback_combo", int(r["total_query_budget"])]["audited_successes"]),
             "Sequential_combo": int(lookup["sequential_combo", int(r["total_query_budget"])]["audited_successes"]),
             "Random_mean": float(r["mean_successes"]), "Denominator": int(r["denominator_per_seed"])} for r in random_rows]


def family_table(directory):
    return [{"Model": r["model"], "Strategy": r["strategy"], "Successes": r["successes"],
             "Denominator": r["denominator"], "Logical_queries": r["all_sample_queries_capped"]}
            for r in read_json(Path(directory) / "budget_summary.json") if r["total_query_budget"] == 30]


def conclusions(session, tasks=None):
    tasks = CORE_TASKS if tasks is None else tuple(tasks)
    if (len(set(tasks)) != len(tasks) or set(tasks) - set(TASKS) or not set(CORE_TASKS) <= set(tasks)
            or "transfer" in tasks and "stability" not in tasks):
        raise ValueError("Invalid experiment selection for the conclusion.")
    paths = {task: completed_result(session, task) for task in tasks}
    combo, families = combination_table(paths["combination"])[-1], family_table(paths["families"])
    lines = ["# Conclusions from THIS session's selected native experiments", "",
             "Completed main workflow: real model predictions, full combination controls and the matched family comparison on both detectors.",
             f"Combination, {combo['Budget']} total queries: single {combo['Single']}/{combo['Denominator']}; feedback {combo['Feedback']}/{combo['Denominator']}; sequential combination {combo['Sequential_combo']}/{combo['Denominator']}; random mean {combo['Random_mean']}/{combo['Denominator']}.", ""]
    for name in ("piguard", "protectai"):
        rows = [r for r in families if r["Model"] == name and r["Strategy"].endswith("_feedback")]
        best_coverage = max(r["Successes"] for r in rows)
        winners = [r for r in rows if r["Successes"] == best_coverage]
        best_cost = min(r["Logical_queries"] for r in winners)
        best = [r["Strategy"] for r in winners if r["Logical_queries"] == best_cost]
        lines.append(f"{name}: best coverage among the three feedback families = {best_coverage}/{rows[0]['Denominator']}; lowest query cost among coverage-tied families = {best_cost}, strategies {best}.")
    if "gradient" in paths:
        grad = gradient_table(paths["gradient"])[-1]
        lines.extend(["", f"Optional white-box, {grad['Budget']} candidates: gradient {grad['Gradient']}/{grad['Denominator']}; sequential {grad['Sequential']}/{grad['Denominator']}; random mean {grad['Random_mean']}/{grad['Denominator']}.",
                      "Ranking has extra forward/backward cost. Better candidate coverage does not by itself establish lower total runtime."])
    if "selector" in paths:
        lines.extend(["", "Optional paid-probe selector outcomes (same-budget comparison):"])
        for row in read_json(paths["selector"] / "criteria.json"):
            lines.append(f"{row['model']}: fixed zero-width {row['zero_successes']}/{row['denominator']} at {row['zero_queries']} queries; selector {row['selector_successes']}/{row['denominator']} at {row['selector_queries']} queries.")
    if "stability" in paths:
        lines.extend(["", "Optional full-pool benign false blocks and repair controls:"])
        for row in csv_rows(paths["stability"] / "family_budget_summary.csv"):
            if row["dataset"] == "benign" and row["budget"] == "all":
                lines.append(f"{row['family']}: {row['new_error_samples']}/{row['baseline_correct']} originally correct benign prompts have a new error; after NFKC, {row['nfkc_postrepair_any_error_samples']} have any tested error; after targeted repair, {row['mixed_script_postrepair_any_error_samples']}.")
    if "transfer" in paths:
        lines.extend(["", "Optional direct frozen-string transfer (text subset):"])
        for row in csv_rows(paths["transfer"] / "transfer_summary.csv"):
            if row["dataset"] == "text" and row["scope"] == "all":
                lines.append(f"{row['family']}: target successes {row['transfer_success']}/{row['transfer_eligible']} eligible source-successful variants.")
    skipped = [task for task in OPTIONAL_TASKS if task not in paths]
    if skipped:
        lines.extend(["", "Optional experiments not executed in the selected workflow: " + ", ".join(skipped) + ". No results or findings are claimed for these sections."])
    lines.extend(["", "Every numerical finding above is read from the selected completed native stages, not copied historical numbers.",
                  "These are already-explored benchmark cohorts, not a held-out generalization study. Preserve negative outcomes.",
                  "Detector flips do not prove downstream execution or semantic preservation. Repairs are limited controls, not a universal defense.",
                  "Independent target search is not the same experiment as direct transfer of a frozen source-successful string.",
                  "No model training, threshold fitting, or weight updates were performed."])
    return "\n".join(lines)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--worker", choices=("baseline", "families"), required=True)
    parser.add_argument("--root", type=Path, required=True)
    parser.add_argument("--checkpoint", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    verify_bundle(args.root)
    (baseline_worker if args.worker == "baseline" else family_worker)(args.root, args.checkpoint, args.output)


if __name__ == "__main__":
    main()


## 4. Set up the native model environment and checkpoint

Use a separate Python 3.10 environment with Torch 2.4.0 and Transformers 4.44.0.
The checkpoint download is verified by SHA256 before any comparison. The selected GPU is tested with an actual tensor operation.
Dependencies/checkpoint installation is a required stage here, not an optional appendix.


In [ ]:
from m3_submission import prepare_native_environment
display(Code(inspect.getsource(prepare_native_environment), language="python"))
# Reuse a verified checkpoint from the earlier full version if it exists in this runtime.
import shutil
previous_checkpoint = Path("/content/FIT5230_M3_Full_Native/checkpoints/best_model.pth")
new_checkpoint = ROOT / "checkpoints/best_model.pth"
if not new_checkpoint.exists() and previous_checkpoint.is_file():
    if hashlib.sha256(previous_checkpoint.read_bytes()).hexdigest() == "ad7eed71bedf2d445a4d452d781bccab6b2101675c2b4f65edae86d444ca1c99":
        new_checkpoint.parent.mkdir(exist_ok=True)
        shutil.copyfile(previous_checkpoint, new_checkpoint)
NATIVE_PYTHON, CHECKPOINT = prepare_native_environment(ROOT)
print("Native Python:", NATIVE_PYTHON)
print("Verified author checkpoint:", CHECKPOINT)


### 4.1 Main workflow and optional experiments

Leave all four switches `False` for the normal main run. The model, combination controls and two-model family comparison still execute.
Change only the extensions you want to `True`; then rerun this options cell and those experiment sections.
Direct transfer also requires `RUN_STABILITY=True`, because it freezes inputs from this session's stability results.
Switches do not change sample counts, mutation rules or per-method budgets inside any enabled experiment.


In [ ]:
from m3_full_runtime import (
    TASKS, CORE_TASKS, OPTIONAL_TASKS, selected_tasks, verify_bundle, run_stage, completed_result, read_json, csv_rows,
    gradient_table, combination_table, family_table, conclusions,
)
verify_bundle(ROOT)
RUN_GRADIENT = False  #@param {type:"boolean"}
RUN_SELECTOR = False  #@param {type:"boolean"}
RUN_STABILITY = False  #@param {type:"boolean"}
RUN_TRANSFER = False  #@param {type:"boolean"}
ACTIVE_TASKS = selected_tasks(dict(gradient=RUN_GRADIENT, selector=RUN_SELECTOR, stability=RUN_STABILITY, transfer=RUN_TRANSFER))
RUN_NAME = "run_1"  # Use a NEW name for an intentionally independent rerun.
assert RUN_NAME.replace("_", "").replace("-", "").isalnum()
SESSION = ROOT / "native_runs" / RUN_NAME
RETRY_INCOMPLETE = False  # True explicitly restarts an incomplete stage; partial files remain.
import matplotlib.pyplot as plt
FIGURES = SESSION / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("Current native session:", SESSION)
show_table([dict(Experiment=task, Mode="Main" if task in CORE_TASKS else "Optional", Enabled=task in ACTIVE_TASKS) for task in TASKS])
print("Only this session's completed native outputs feed the result cells below.")


## 5. Data, labels and fixed evaluation scope

The GitHub data contains 75 BIPIA text payloads, 50 code payloads and 339 NotInject benign prompts.
All 464 originals are used only when the optional full stability experiment is enabled.
The other protocols preserve their previously fixed eligible cohorts: 22 PIGuard texts and 13 ProtectAI texts.
Eligibility is checked against the original predictions; existing errors do not count as newly induced successes.
These benchmarks have already been explored; this is not a new held-out dataset.


In [ ]:
from collections import Counter
import eval_m3_unicode_stability as stability_core
records = stability_core.load_records(ROOT / "datasets")
print("Available source data below; the main protocols use their fixed 22/13-text cohorts, not this full pool.")
show_table([dict(Dataset=k, Originals=v) for k, v in Counter(r["dataset"] for r in records).items()])
if "stability" in ACTIVE_TASKS:
    stability_plan = stability_core.preview(records)
    show_table(stability_plan)
    print("Planned unique model calls in enabled full stability stage:", sum(r["planned_unique_model_calls"] for r in stability_plan))
else:
    print("Full stability/repair is disabled: its 16,565 unique model inputs will NOT be evaluated.")
print("Labels stay unchanged. A detector miss is not proof of downstream instruction execution.")


## 6. Load the model and obtain real original/modified predictions

This cell really instantiates PIGuard and loads the author weights.
The example shows the original text and three Unicode variants with actual labels and scores.
It is an illustration, not the benchmark success rate. A successful illustration is not required to continue.


In [ ]:
# Real inference. A completed stage from THIS session is reused explicitly on a rerun.
BASELINE_RESULT = run_stage(ROOT, SESSION, NATIVE_PYTHON, CHECKPOINT, "baseline", RETRY_INCOMPLETE)


In [ ]:
baseline_manifest = read_json(BASELINE_RESULT / "manifest.json")
show_table([{k: baseline_manifest[k] for k in ("model", "device", "parameters", "native_window", "parameter_updates")}])
show_table(read_json(BASELINE_RESULT / "predictions.json"))


## 7. Optional A: word selection under a fixed candidate budget

OFF by default. Enable `RUN_GRADIENT=True` in Section 4.1 to run this entire protocol.

Run all 22 eligible text originals with sequential ordering, gradient ordering and ten pre-fixed random seeds (5230–5239).
Each method receives at most ten one-character candidates per text. Seeds are separate controls, not a pooled 100-query attack.
Gradient ranking has additional forwards/backwards; compare candidate coverage separately from total compute.


In [ ]:
if "gradient" in ACTIVE_TASKS:
    word_plan = read_json(ROOT / "plans/text_plan.json")
    print("Fixed eligible texts:", len(word_plan["selected"]))
    print("Candidate budgets:", word_plan["config"]["budgets"])
    print("Pre-fixed random seeds:", word_plan["config"]["random_seeds"])
    print("Cohort is frozen before THIS run's scores; no success-based resampling.")
else:
    print("Optional gradient not enabled: no current-run result is claimed here.")


In [ ]:
if "gradient" in ACTIVE_TASKS:
    # Real inference. A completed stage from THIS session is reused explicitly on a rerun.
    GRADIENT_RESULT = run_stage(ROOT, SESSION, NATIVE_PYTHON, CHECKPOINT, "gradient", RETRY_INCOMPLETE)
else:
    GRADIENT_RESULT = None
    print("Skipped optional gradient; complete code remains available. No model call made.")


In [ ]:
if "gradient" in ACTIVE_TASKS:
    gradient_results = gradient_table(GRADIENT_RESULT)
    show_table(gradient_results)
    print("Extra gradient costs are recorded in the native manifest:")
    gradient_manifest = read_json(GRADIENT_RESULT / "manifest.json")
    show_table([{k: gradient_manifest[k] for k in ("device", "original_native_forward_passes", "gradient_ranking_forward_passes", "gradient_ranking_backward_passes", "parameter_updates")}])
else:
    print("Optional gradient not enabled: no current-run result is claimed here.")


In [ ]:
if "gradient" in ACTIVE_TASKS:
    import matplotlib.pyplot as plt
    FIGURES = SESSION / "figures"
    FIGURES.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    for key in ("Sequential", "Gradient", "Random_mean"):
        ax.plot([r["Budget"] for r in gradient_results], [r[key] for r in gradient_results], marker="o", label=key)
    ax.set(xlabel="Candidate queries per sample", ylabel="New detector misses", title="Fresh white-box ranking comparison")
    ax.legend(); ax.grid(alpha=.2)
    fig.savefig(FIGURES / "gradient.png", dpi=160)
    plt.show(); plt.close(fig)
else:
    print("Optional gradient not enabled: no current-run result is claimed here.")


## 8. Experiment B: single edits versus joint edits

Run the full original 22-text protocol: single, feedback combination, sequential combination and ten fixed random controls.
Each has at most 30 total calls including the original and reduction checks; an input has at most three edited characters.
The early-budget comparison and final coverage are different questions. A finite-budget failure is not an exhaustive impossibility claim.


In [ ]:
# Real inference. A completed stage from THIS session is reused explicitly on a rerun.
COMBINATION_RESULT = run_stage(ROOT, SESSION, NATIVE_PYTHON, CHECKPOINT, "combination", RETRY_INCOMPLETE)


In [ ]:
combination_results = combination_table(COMBINATION_RESULT)
show_table(combination_results)
print("All original checks, failed probes and reduction calls are included in total-query accounting.")


In [ ]:
outcomes = read_json(COMBINATION_RESULT / "outcomes.json")
observed = [r for r in outcomes if r["strategy"] == "feedback_combo" and r["first_success_audited"]]
show_table(observed[:2])
print("These are newly observed examples. Inspect actual text and visibility before interpreting them.")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
for key in ("Single", "Feedback", "Sequential_combo", "Random_mean"):
    ax.plot([r["Budget"] for r in combination_results], [r[key] for r in combination_results], marker="o", label=key)
ax.set(xlabel="Total oracle queries per sample", ylabel="Audited new misses", title="Fresh combination comparison")
ax.legend(); ax.grid(alpha=.2)
fig.savefig(FIGURES / "combination.png", dpi=160)
plt.show(); plt.close(fig)


## 9. Experiment C: matched character families on two detectors

Run Cyrillic, zero-width and mixed search, each with feedback and sequential control.
Both PIGuard and the pinned ProtectAI detector are loaded and queried in THIS run, one model at a time.
Each model retains its own eligible cohort, original weights, native window (256/512) and argmax rule.
Do not rank model strength by comparing the different 22/13 denominators.


In [ ]:
family_plan = read_json(ROOT / "plans/family_plan.json")
show_table([
    dict(Model="PIGuard", Eligible_texts=len(family_plan["source_selected"]), Native_window=256),
    dict(Model="ProtectAI", Eligible_texts=len(family_plan["target_selected"]), Native_window=512),
])
print("Six methods; at most three unit edits and 30 total queries per method/sample.")
print("U+200B is a zero-width Unicode character, not an ordinary space.")


In [ ]:
# Real inference. A completed stage from THIS session is reused explicitly on a rerun.
FAMILIES_RESULT = run_stage(ROOT, SESSION, NATIVE_PYTHON, CHECKPOINT, "families", RETRY_INCOMPLETE)


In [ ]:
family_results = family_table(FAMILIES_RESULT)
show_table(family_results)
print("Both models have fresh inference; no old ProtectAI attack outcomes are reused.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, model_name in zip(axes, ("piguard", "protectai")):
    rows = [r for r in family_results if r["Model"] == model_name and r["Strategy"].endswith("_feedback")]
    bars = ax.bar([r["Strategy"].replace("_feedback", "") for r in rows], [r["Logical_queries"] for r in rows])
    for bar, row in zip(bars, rows):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f"{row['Successes']}/{row['Denominator']}", ha="center", va="bottom")
    ax.set(title=model_name, ylabel="Total logical queries (including failures)")
fig.savefig(FIGURES / "family_costs.png", dpi=160)
plt.show(); plt.close(fig)


## 10. Optional D: does automatic family selection justify its probe cost?

OFF by default. Enable `RUN_SELECTOR=True` in Section 4.1.

Run the original four-method selector experiment on both models with new predictions.
Compare fixed zero-width, fixed Cyrillic, lower-score selection, and same-probe/always-zero continuation.
Success coverage must not be sacrificed to claim a query saving. Keep negative outcomes as results.


In [ ]:
if "selector" in ACTIVE_TASKS:
    # Real inference. A completed stage from THIS session is reused explicitly on a rerun.
    SELECTOR_RESULT = run_stage(ROOT, SESSION, NATIVE_PYTHON, CHECKPOINT, "selector", RETRY_INCOMPLETE)
else:
    SELECTOR_RESULT = None
    print("Skipped optional selector; complete code remains available. No model call made.")


In [ ]:
if "selector" in ACTIVE_TASKS:
    selector_results = read_json(SELECTOR_RESULT / "criteria.json")
    show_table(selector_results)
    print("Read both coverage and total queries; a more complicated policy is not automatically better.")
else:
    print("Optional selector not enabled: no current-run result is claimed here.")


## 11. Optional E: full-pool stability and repair controls

OFF by default. Enable `RUN_STABILITY=True` in Section 4.1.

Run ALL 464 originals and ALL 16,101 Unicode variants plus the original repair controls.
This is the largest stage. There is no smoke-only substitution.
Evaluate injection misses and benign false blocks separately. A new-error-exists rate is not a deployment false-positive rate.
Families share candidate word occurrences, but their character-edit counts differ in this full-pool protocol.


In [ ]:
# Repair receives only the current string, never the original answer or label.
display(Code(inspect.getsource(stability_core.repair), language="python"))


In [ ]:
if "stability" in ACTIVE_TASKS:
    # Real inference. A completed stage from THIS session is reused explicitly on a rerun.
    STABILITY_RESULT = run_stage(ROOT, SESSION, NATIVE_PYTHON, CHECKPOINT, "stability", RETRY_INCOMPLETE)
else:
    STABILITY_RESULT = None
    print("Skipped optional stability; complete code remains available. No model call made.")


In [ ]:
if "stability" in ACTIVE_TASKS:
    clean_results = csv_rows(STABILITY_RESULT / "clean_summary.csv")
    stability_results = [r for r in csv_rows(STABILITY_RESULT / "family_budget_summary.csv") if r["budget"] == "all"]
    show_table(clean_results)
    show_table(stability_results)
    print("Report ordinary normalization and targeted repair separately; neither is a universal defense guarantee.")
else:
    print("Optional stability not enabled: no current-run result is claimed here.")


In [ ]:
if "stability" in ACTIVE_TASKS:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, dataset in zip(axes, ("text", "code", "benign")):
        rows = [r for r in stability_results if r["dataset"] == dataset]
        ax.bar([r["family"] for r in rows], [int(r["new_error_samples"]) for r in rows])
        ax.set(title=dataset, ylabel="Originally correct prompts with a new error")
        ax.tick_params(axis="x", rotation=20)
    fig.savefig(FIGURES / "two_way_errors.png", dpi=160)
    plt.show(); plt.close(fig)
else:
    print("Optional stability not enabled: no current-run result is claimed here.")


### Actual new-run examples

Show a first observed error for each dataset if one exists. `[U+200B]` is a display marker only.
Character recovery and visible offsets do not prove that the modified instruction retains its function; semantic review remains a limitation.


In [ ]:
if "stability" in ACTIVE_TASKS:
    all_attempts = csv_rows(STABILITY_RESULT / "attempts.csv")
    examples = []
    for dataset in ("text", "code", "benign"):
        row = next((r for r in all_attempts if r["dataset"] == dataset and r["new_error"] == "1"), None)
        if row:
            examples.append({k: row[k] for k in ("sample_id", "dataset", "family", "label", "prediction", "truncated", "edited_span_visibility")})
            print("Input:", row["text"].replace("​", "[U+200B]"))
            print("Escaped:", row["text"].encode("unicode_escape").decode())
    show_table(examples)
else:
    print("Optional stability not enabled: no current-run result is claimed here.")


## 12. Optional F: direct transfer of frozen successful strings

OFF by default. Enable both `RUN_TRANSFER=True` and `RUN_STABILITY=True` in Section 4.1.

Freeze source choices using THIS session's completed PIGuard stability run before querying the target.
Then load ProtectAI and predict all original/frozen variant inputs required by the unchanged transfer protocol.
This is different from Experiment C, where the attacker independently searches each model.
No target feedback is used to select the source-successful strings for direct transfer.


In [ ]:
if "transfer" in ACTIVE_TASKS:
    # Real inference. A completed stage from THIS session is reused explicitly on a rerun.
    TRANSFER_RESULT = run_stage(ROOT, SESSION, NATIVE_PYTHON, CHECKPOINT, "transfer", RETRY_INCOMPLETE)
else:
    TRANSFER_RESULT = None
    print("Skipped optional transfer; complete code remains available. No model call made.")


In [ ]:
if "transfer" in ACTIVE_TASKS:
    transfer_results = [r for r in csv_rows(TRANSFER_RESULT / "transfer_summary.csv") if r["scope"] == "all"]
    show_table(transfer_results)
    print("Small eligible subsets cannot establish that transfer is generally possible or impossible.")
else:
    print("Optional transfer not enabled: no current-run result is claimed here.")


## 13. Conclusions generated from the current native results

The following cell requires all main stages and all extensions explicitly enabled in Section 4.1 to complete.
It reads only those current-session results. Disabled extensions are listed as not executed, without a result claim—even if older optional files exist in this session.
It never prints a fixed historical success count as a new result.
If a new result differs from the earlier work, report the difference rather than replacing it.


In [ ]:
native_conclusions = conclusions(SESSION, ACTIVE_TASKS)
display(Markdown(native_conclusions))
(SESSION / "CONCLUSIONS_EN.md").write_text(native_conclusions, encoding="utf-8")


## 14. Limitations and interpretation

- Previously explored author benchmarks and frozen cohorts, not independent held-out evaluation; training overlap is unknown.
- Payload-only labels are kept unchanged. Detector misses do not establish downstream prompt-injection execution or preserved semantics.
- A Unicode replacement is not necessarily invisible to a tokenizer. Truncation/offset audits are necessary but not semantic proof.
- White-box gradients and score-visible black-box feedback are different access assumptions. Gradient computation is not free.
- Shared physical caches save inference, but every method's logical calls still count, including failures and reductions.
- Native model windows differ. Compare strategies within each fixed cohort; do not turn different denominators into model-strength rankings.
- Mixed edits, family selection and repair are hypotheses with bounded evidence, not guaranteed improvements.
- We do not retrain the detector or change its checkpoint, labels, threshold or native decision rule to manufacture a gain.

Before submission, fill in the real individual contributions at the top and make the Colab viewable to the marker.


In [ ]:
verify_bundle(ROOT)
completion = []
for task in TASKS:
    if task in ACTIVE_TASKS:
        directory = completed_result(SESSION, task)
        completion.append(dict(Stage=task, Status="Completed", Result=str(directory)))
    else:
        completion.append(dict(Stage=task, Status="Optional: not executed", Result="No current-run finding claimed"))
show_table(completion)
print("All selected stages produced native results. Save this Colab WITH its outputs, including explicit skip notices.")


## 15. Save the notebook; optional evidence backup

The submitted artifact is the Colab link in the required PDF/text file. Keep all relevant cell outputs visible and saved in Colab.
You do NOT have to download a ZIP to make this notebook work or to view the results.
The optional switch below makes a portable evidence backup, excluding model weights and installed environments. It is off by default.


In [ ]:
DOWNLOAD_BACKUP = False
if DOWNLOAD_BACKUP:
    import zipfile
    package = SESSION.parent / (SESSION.name + "_evidence.zip")
    with zipfile.ZipFile(package, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for folder in (SOURCE, ROOT / "datasets", ROOT / "plans", SESSION):
            for path in sorted(folder.rglob("*")):
                if path.is_file() and "__pycache__" not in path.parts:
                    archive.write(path, path.relative_to(ROOT))
        archive.write(ROOT / "inventory.json", "inventory.json")
        archive.write(ROOT / "LICENSE", "LICENSE")
    from google.colab import files
    files.download(str(package))
else:
    print("No download requested. Save the notebook with outputs; the evidence ZIP is optional.")


## References and reproducibility

- [Original PIGuard implementation](https://github.com/leolee99/PIGuard)
- [Our M3 source and data](https://github.com/dcy1279-hash/PIGuard/tree/main/m3) are downloaded at the fixed source commit recorded in the setup cell.
- ProtectAI checkpoint: `protectai/deberta-v3-base-prompt-injection-v2`, revision `90c9989b1a342275dd0d1a95aad283c04e075671`.
- [Unicode normalization](https://www.unicode.org/reports/tr15/)
- `inventory.json` records source/data/plan hashes. `native_runs/<RUN_NAME>/runs.json` records completed fresh stages and result-file hashes.
- The earlier archived-analysis notebook remains a separate historical record; this workbook does not overwrite it.
